# 02 · Attention

Follow how a shared weight matrix turns token embeddings into queries, down to
the individual multiplications. Then move query and key vectors with sliders
to see how their dot products measure a match.

## 1. Start with token embeddings

Each token starts with an embedding: a row of numbers from the embedding table.
Collecting those rows gives us a matrix called $X$, with one row per token and
one column per embedding coordinate.

The sentence below has 20 tokens. Each embedding has four coordinates, so
$X$ has shape $(20, 4)$.


In [1]:
from pathlib import Path

import numpy as np
from IPython.display import Math, display
from model_tokenizer import Tokenizer

# Recreate the same token embeddings as notebook 01.
tokenizer = Tokenizer.load(
    Path("../../../trained-models/tokenizers/simple-english.json").resolve()
)
text = "Once upon a time, there was a fluffy bunny."
token_ids = np.array(tokenizer.encode(text), dtype=np.int64)
embedding_dim = 4
embedding_rng = np.random.default_rng(42)
embedding_table = embedding_rng.normal(
    loc=0.0, scale=0.02, size=(tokenizer.vocab_size, embedding_dim)
).astype(np.float32)
X = embedding_table[token_ids]

print(f"X: {X.shape[0]} token rows × {X.shape[1]} embedding coordinates")


X: 20 token rows × 4 embedding coordinates


## 2. Turn each embedding into a query

Attention uses a **query vector** for each token to determine which tokens to
draw information from. It compares that query with vectors called **keys**
associated with the tokens it can attend to.

To produce a query, we mix the coordinates of a token's embedding using
weighted sums. This transformation is called a **linear projection**.
Its weights form a matrix called $W_Q$: $W$ stands for weights, and the
subscript $Q$ identifies their job as producing queries.

$W_Q$ is a **single shared, learned transformation**, applied independently to
every token's embedding. Training adjusts this shared matrix; different token
embeddings pass through the same transformation to produce their query vectors.

We call the resulting matrix of query vectors $Q$:

$$
\underbrace{Q}_{\text{query vectors}}
=
\underbrace{X}_{\text{token embeddings}}
\underbrace{W_Q}_{\text{query weights}}
$$

Here, $W_Q$ has four input rows and four output columns. Each column tells us
how to combine the four embedding coordinates into one query coordinate:

$$
\underbrace{(20 \times 4)}_{X}
\;\underbrace{(4 \times 4)}_{W_Q}
\;\longrightarrow\;\underbrace{(20 \times 4)}_{Q}
$$

In NumPy, we write this matrix multiplication as `Q = X @ W_Q`.
The weights below start as random numbers, so the resulting queries have not
yet learned which information is useful.


In [2]:
# Initialize query weights separately from the embedding table.
# A fixed seed gives us the same initial weights on each run.
query_rng = np.random.default_rng(7)
W_Q = query_rng.normal(
    loc=0.0, scale=0.5, size=(embedding_dim, embedding_dim)
).astype(np.float32)

# @ is matrix multiplication, not element-by-element multiplication.
Q = X @ W_Q
print(f"Q shape: {Q.shape}")


Q shape: (20, 4)


In [3]:
def latex_matrix(values):
    """Format a small vector or matrix for notebook math display."""
    rows = np.atleast_2d(values)
    body = r" \\ ".join(
        " & ".join(f"{float(value):.5f}" for value in row)
        for row in rows
    )
    return r"\begin{bmatrix}" + body + r"\end{bmatrix}"


display(Math(r"W_Q = " + latex_matrix(W_Q)))


<IPython.core.display.Math object>

## 3. Follow one token through the projection

The first token is **Once**. Its embedding is the first row of $X$; its query
is the first row of $Q$.

$$\mathbf q = \mathbf x W_Q$$

We display that row calculation below. Numbers are rounded for readability;
NumPy uses the stored, unrounded numbers.


In [4]:
x = X[0]
q = Q[0]


In [5]:
display(Math(
    latex_matrix(x) + r"\;" + latex_matrix(W_Q)
    + r"\;=\;" + latex_matrix(q)
))


<IPython.core.display.Math object>

## 4. Expand one weighted sum

To get the **first** query coordinate, multiply each embedding coordinate by
the corresponding entry in the **first column** of $W_Q$, then add:

$$q_1 = x_1 w_{11} + x_2 w_{21} + x_3 w_{31} + x_4 w_{41}$$

The mathematical labels above start at 1. Python indices start at 0, so this
same coordinate is `q[0]` and that column is `W_Q[:, 0]`.


In [6]:
# Expose the four products that make the first query coordinate.
first_column = W_Q[:, 0]
products = x * first_column


In [7]:
expanded_terms = " + ".join(
    rf"({float(value):.5f} \times {float(weight):.5f})"
    for value, weight in zip(x, first_column)
)
product_terms = " + ".join(f"({float(value):.7f})" for value in products)
display(Math(r"q_1 = " + expanded_terms))
display(Math(r"q_1 \approx " + product_terms))
display(Math(rf"q_1 \approx {float(q[0]):.7f}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 5. Explore how a weight affects the result

Each of the other three query coordinates uses another column of $W_Q$ in the
same way. No token has looked at another token yet: this operation transforms
each row independently.

Try changing one entry in $W_Q$ **after its initialization**, then rerun the
projection and calculation cells. Changing $W_Q[0, 0]$ changes how much the
first input coordinate contributes to the first query coordinate, for every
token. Rerunning the initialization cell resets your edit.

Trace the four multiplications in $q_1$. Which numbers come from the token's
embedding, which are trainable weights, and which number is the result?


## 6. Compare a query with keys

A query needs something to match against. Each token also has a **key vector**: the query and key are the two sides of a comparison.

For this comparison, use two coordinates so we can draw each vector as an arrow. A **dot product** multiplies corresponding coordinates and adds the products:

$$\text{score} = \mathbf q \cdot \mathbf k = q_x k_x + q_y k_y$$

A positive score means the vectors point broadly together; perpendicular vectors score zero; opposing directions give a negative score. Longer vectors can produce larger scores too, so this is not just an angle measurement.


In [8]:
# Compare one query with two candidate keys.
example_query = np.array([1.0, 0.0])
example_keys = np.array([[1.0, 1.0], [-1.0, 1.0]])

# Each key row produces one score against the same query.
example_scores = example_keys @ example_query
print("Scores for Key A and Key B:", example_scores)


Scores for Key A and Key B: [ 1. -1.]


### Move the vectors

The sliders start with the same numbers as the calculation above and update the arrows and scores immediately. Blue is the query, orange is Key A, and purple is Key B.

Keep the query at **(1, 0)** and try these changes to Key A:

1. **(1, 0)**: the key points along the query.
2. **(0, 1)**: the key is perpendicular to the query.
3. **(−1, 0)**: the key points against the query.
4. **(2, 0)**: the direction stays the same, but the length doubles.

Then change the query: both comparisons now change because they share that query. A zero query gives zero for both scores. These scores are raw numbers, not probabilities.


In [9]:
from html import escape

# Keep this interactive figure isolated from the notebook page.
vector_page = "<!doctype html><html><head><meta charset=\"utf-8\"><style>body{font:15px system-ui;margin:12px;color:#172033;background:#fff} .layout{display:flex;gap:24px;flex-wrap:wrap}svg{width:360px;max-width:100%;height:360px}label{display:grid;grid-template-columns:55px 160px 45px;align-items:center;gap:8px;margin:10px 0}fieldset{border:1px solid #ccd3df;border-radius:8px;margin-bottom:12px}output{font-variant-numeric:tabular-nums}.score{padding:12px;border-left:4px solid;margin:12px 0;background:#f3f5f9}button{padding:6px 10px;margin-right:6px}#summary{max-width:680px}</style></head><body><div class=\"layout\"><div><svg viewBox=\"0 0 360 360\" role=\"img\" aria-label=\"Query and key vectors on coordinate axes\"><defs><marker id=\"qhead\" markerWidth=\"8\" markerHeight=\"8\" refX=\"7\" refY=\"4\" orient=\"auto\"><path d=\"M0 0 L8 4 L0 8\" fill=\"#2563eb\"/></marker><marker id=\"ahead\" markerWidth=\"8\" markerHeight=\"8\" refX=\"7\" refY=\"4\" orient=\"auto\"><path d=\"M0 0 L8 4 L0 8\" fill=\"#d97706\"/></marker><marker id=\"bhead\" markerWidth=\"8\" markerHeight=\"8\" refX=\"7\" refY=\"4\" orient=\"auto\"><path d=\"M0 0 L8 4 L0 8\" fill=\"#9333ea\"/></marker></defs><g id=\"grid\"></g><line x1=\"15\" y1=\"180\" x2=\"345\" y2=\"180\" stroke=\"#64748b\"/><line x1=\"180\" y1=\"15\" x2=\"180\" y2=\"345\" stroke=\"#64748b\"/><text x=\"338\" y=\"198\">x</text><text x=\"162\" y=\"18\">y</text><g id=\"arrows\"></g></svg><p>All arrows start at (0, 0). Each grid step is 1.</p></div><div id=\"controls\"></div></div><div id=\"scores\" aria-live=\"polite\"></div><p id=\"summary\">These are raw match scores, not probabilities. Both direction and length affect the dot product.</p><button id=\"reset\">Reset vectors</button><script>\nconst colors={q:'#2563eb',a:'#d97706',b:'#9333ea'};\nconst names={q:'Query',a:'Key A',b:'Key B'};\nconst initial={qx:1,qy:0,ax:1,ay:1,bx:-1,by:1};\nconst grid=document.getElementById('grid');\nfor(let n=-2;n<=2;n++){const p=180+n*65;grid.innerHTML+='<line x1=\"'+p+'\" y1=\"20\" x2=\"'+p+'\" y2=\"340\" stroke=\"#e2e8f0\"/><line x1=\"20\" y1=\"'+p+'\" x2=\"340\" y2=\"'+p+'\" stroke=\"#e2e8f0\"/>';}\nfor(const id of ['q','a','b']){const box=document.createElement('fieldset');box.innerHTML='<legend style=\"color:'+colors[id]+'\">'+names[id]+'</legend>';for(const axis of ['x','y']){const key=id+axis;box.innerHTML+='<label for=\"'+key+'\">'+axis+' <input id=\"'+key+'\" aria-label=\"'+names[id]+' '+axis+'\" type=\"range\" min=\"-2\" max=\"2\" step=\"0.1\" value=\"'+initial[key]+'\"><output id=\"'+key+'-value\"></output></label>';}document.getElementById('controls').append(box);}\nfunction update(){const v={};for(const key in initial){v[key]=Number(document.getElementById(key).value);document.getElementById(key+'-value').textContent=v[key].toFixed(1);}document.getElementById('arrows').innerHTML=['q','a','b'].map(id=>{const x=180+65*v[id+'x'],y=180-65*v[id+'y'];return '<line x1=\"180\" y1=\"180\" x2=\"'+x+'\" y2=\"'+y+'\" stroke=\"'+colors[id]+'\" stroke-width=\"3\" marker-end=\"url(#'+(id==='q'?'q':id)+'head)\"/><text x=\"'+(x+8)+'\" y=\"'+(y-8)+'\" fill=\"'+colors[id]+'\">'+names[id]+'</text>';}).join('');document.getElementById('scores').innerHTML=['a','b'].map(id=>{const p1=v.qx*v[id+'x'],p2=v.qy*v[id+'y'],score=p1+p2;return '<div class=\"score\" style=\"border-color:'+colors[id]+'\"><strong>Query · '+names[id]+' = '+score.toFixed(2)+'</strong><br>('+v.qx.toFixed(1)+' × '+v[id+'x'].toFixed(1)+') + ('+v.qy.toFixed(1)+' × '+v[id+'y'].toFixed(1)+') = '+p1.toFixed(2)+' + ('+p2.toFixed(2)+')</div>';}).join('');}\ndocument.querySelectorAll('input').forEach(input=>input.addEventListener('input',update));document.getElementById('reset').onclick=()=>{for(const key in initial)document.getElementById(key).value=initial[key];update();};update();\n</script></body></html>"
display({"text/html":
    '<iframe title="Interactive query and key comparison" '
    'sandbox="allow-scripts" style="width:100%;height:690px;border:0" '
    'srcdoc="' + escape(vector_page, quote=True) + '"></iframe>'
}, raw=True)


<iframe title="Interactive query and key comparison" sandbox="allow-scripts" style="width:100%;height:690px;border:0" srcdoc="<!doctype html><html><head><meta charset="utf-8"><style>body{font:15px system-ui;margin:12px;color:#172033;background:#fff} .layout{display:flex;gap:24px;flex-wrap:wrap}svg{width:360px;max-width:100%;height:360px}label{display:grid;grid-template-columns:55px 160px 45px;align-items:center;gap:8px;margin:10px 0}fieldset{border:1px solid #ccd3df;border-radius:8px;margin-bottom:12px}output{font-variant-numeric:tabular-nums}.score{padding:12px;border-left:4px solid;margin:12px 0;background:#f3f5f9}button{padding:6px 10px;margin-right:6px}#summary{max-width:680px}</style></head><body><div class="layout"><div><svg viewBox="0 0 360 360" role="img" aria-label="Query and key vectors on coordinate axes"><defs><marker id="qhead" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0 0 L8 4 L0 8" fill="#2563eb"/></marker><marker id="ahead" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0 0 L8 4 L0 8" fill="#d97706"/></marker><marker id="bhead" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0 0 L8 4 L0 8" fill="#9333ea"/></marker></defs><g id="grid"></g><line x1="15" y1="180" x2="345" y2="180" stroke="#64748b"/><line x1="180" y1="15" x2="180" y2="345" stroke="#64748b"/><text x="338" y="198">x</text><text x="162" y="18">y</text><g id="arrows"></g></svg><p>All arrows start at (0, 0). Each grid step is 1.</p></div><div id="controls"></div></div><div id="scores" aria-live="polite"></div><p id="summary">These are raw match scores, not probabilities. Both direction and length affect the dot product.</p><button id="reset">Reset vectors</button><script>
const colors={q:'#2563eb',a:'#d97706',b:'#9333ea'};
const names={q:'Query',a:'Key A',b:'Key B'};
const initial={qx:1,qy:0,ax:1,ay:1,bx:-1,by:1};
const grid=document.getElementById('grid');
for(let n=-2;n<=2;n++){const p=180+n*65;grid.innerHTML+='<line x1="'+p+'" y1="20" x2="'+p+'" y2="340" stroke="#e2e8f0"/><line x1="20" y1="'+p+'" x2="340" y2="'+p+'" stroke="#e2e8f0"/>';}
for(const id of ['q','a','b']){const box=document.createElement('fieldset');box.innerHTML='<legend style="color:'+colors[id]+'">'+names[id]+'</legend>';for(const axis of ['x','y']){const key=id+axis;box.innerHTML+='<label for="'+key+'">'+axis+' <input id="'+key+'" aria-label="'+names[id]+' '+axis+'" type="range" min="-2" max="2" step="0.1" value="'+initial[key]+'"><output id="'+key+'-value"></output></label>';}document.getElementById('controls').append(box);}
function update(){const v={};for(const key in initial){v[key]=Number(document.getElementById(key).value);document.getElementById(key+'-value').textContent=v[key].toFixed(1);}document.getElementById('arrows').innerHTML=['q','a','b'].map(id=>{const x=180+65*v[id+'x'],y=180-65*v[id+'y'];return '<line x1="180" y1="180" x2="'+x+'" y2="'+y+'" stroke="'+colors[id]+'" stroke-width="3" marker-end="url(#'+(id==='q'?'q':id)+'head)"/><text x="'+(x+8)+'" y="'+(y-8)+'" fill="'+colors[id]+'">'+names[id]+'</text>';}).join('');document.getElementById('scores').innerHTML=['a','b'].map(id=>{const p1=v.qx*v[id+'x'],p2=v.qy*v[id+'y'],score=p1+p2;return '<div class="score" style="border-color:'+colors[id]+'"><strong>Query · '+names[id]+' = '+score.toFixed(2)+'</strong><br>('+v.qx.toFixed(1)+' × '+v[id+'x'].toFixed(1)+') + ('+v.qy.toFixed(1)+' × '+v[id+'y'].toFixed(1)+') = '+p1.toFixed(2)+' + ('+p2.toFixed(2)+')</div>';}).join('');}
document.querySelectorAll('input').forEach(input=>input.addEventListener('input',update));document.getElementById('reset').onclick=()=>{for(const key in initial)document.getElementById(key).value=initial[key];update();};update();
</script></body></html>">